In [1]:
import os, sys
os.chdir("/files/week-03/Project/code")
sys.path.insert(0, os.getcwd())
print(os.getcwd(), os.path.exists("helpers.py"))

/files/week-03/Project/code True


In [2]:
from helpers import *

train, test = load_returns()
print(train.shape, test.shape)

moy = train.groupby("regime_lag")[FACT].mean() * 100
print(moy.round(2))

(437, 5) (320, 5)
            size  value  quality  momentum
regime_lag                                
0          -0.10   0.11     0.16      0.97
1           0.67   0.62    -0.16      0.83
2           1.35   0.33     0.01      1.16
3           0.02   0.63     0.41      0.70


In [3]:
poids = {}
for r in moy.index:
    top2 = moy.loc[r].nlargest(2).index
    w = pd.Series(0.0, index=FACT)
    w[top2] = 0.5
    poids[r] = w

poids = pd.DataFrame(poids).T
print(poids)

   size  value  quality  momentum
0   0.0    0.0      0.5       0.5
1   0.5    0.0      0.0       0.5
2   0.5    0.0      0.0       0.5
3   0.0    0.5      0.0       0.5


In [4]:

def strategie_regle(data):
    W = poids.loc[data["regime_lag"]].values
    return pd.Series((W * data[FACT].values).sum(axis=1), index=data.index)

resultats = {}
for nom, data in [("train", train), ("test", test)]:
    resultats[(nom, "poids egaux")] = perf(data[FACT].mean(axis=1))
    resultats[(nom, "regle par regime")] = perf(strategie_regle(data))

print(pd.DataFrame(resultats).T.round(3))

                        rendement_cumule  sharpe  max_drawdown
train poids egaux                  4.661   1.284        -0.104
      regle par regime            17.757   1.200        -0.109
test  poids egaux                  1.055   0.472        -0.172
      regle par regime             0.963   0.309        -0.353


In [6]:
r_regle = strategie_regle(test)
cum = (1 + r_regle).cumprod()
dd = cum / cum.cummax() - 1

print("Pire drawdown de la règle :", dd.idxmin().date())
print(test["regime_lag"].value_counts().sort_index())
print(perf(train["momentum"]).round(3))
print(perf(test["momentum"]).round(3))
tableau = pd.DataFrame(resultats).T.round(3)
tableau.to_csv("../results/regle_performance.csv")
poids.round(2).to_csv("../results/regle_poids.csv")
print(tableau)

Pire drawdown de la règle : 2009-11-01
regime_lag
0    184
1      3
2     18
3    115
Name: count, dtype: int64
rendement_cumule    37.208
sharpe               0.913
max_drawdown        -0.203
dtype: float64
rendement_cumule    0.250
sharpe              0.141
max_drawdown       -0.578
dtype: float64
                        rendement_cumule  sharpe  max_drawdown
train poids egaux                  4.661   1.284        -0.104
      regle par regime            17.757   1.200        -0.109
test  poids egaux                  1.055   0.472        -0.172
      regle par regime             0.963   0.309        -0.353
